# 命令セットへのコンパイル

この章では、`qret compile` で回路を `SC_LS_FIXED_V0` へ変換する手順を確認します。
生成した pipeline state JSON は、次章の `profile` と可視化で使います。

本章で確認する内容は次の 4 点です。

1. `compile` の基本実行手順
2. `Dim2` / `Dim3` / `DistributedDim2` の違い
3. `PBC` の有効化方法
4. PBC で必要な入力条件（末尾で全量子ビット測定）


In [ ]:
import pathlib
import os
import platform

from IPython.display import Code

project_root = pathlib.Path("../../../..").resolve()
qret_path = project_root / "build" / "main"
if platform.system() == "Darwin": 
    gridsynth_path = project_root / "externals" / "bin" / "gridsynth_macos"
else:
    gridsynth_path = project_root / "externals" / "bin" / "gridsynth"

os.environ["GRIDSYNTH_PATH"] = str(gridsynth_path)
os.environ["PATH"] = str(qret_path) + os.pathsep + os.environ.get("PATH", "")

output_dir = pathlib.Path("../../tutorial-output")
output_dir.mkdir(exist_ok=True)

## 0. 事前確認
最初に CLI が起動できることを確認します。

In [ ]:
!qret --version

## 入力回路（通常モード）
この章の Dim2 / Dim3 / DistributedDim2 では `data/tutorial_5.qasm` を使います。

In [ ]:
tutorial_5_qasm_path = project_root / "quration-docs" / "tutorial" / "data" / "tutorial_5.qasm"
Code(filename=tutorial_5_qasm_path, language="ASM")

## qret compile
`compile` は source (`IR`/`OpenQASM2`/`SC_LS_FIXED_V0`) を読み取り、
対象 ISA (`SC_LS_FIXED_V0`) の命令列へ変換します。

In [ ]:
!qret compile --help

## パイプラインファイルを読む
各 YAML には `source` / `input` / `output` / `sc_ls_fixed_v0_topology` / `sc_ls_fixed_v0_machine_type` / `pass` などの実行条件を定義します。
各モードでファイルを読み分けながら、設定値の差分を確認します。


### Machine Type と PBC モード
- `Dim2`: 単一平面
- `Dim3`: 3D 配置
- `DistributedDim2`: 分散量子計算
- `sc_ls_fixed_v0_enable_pbc_mode` を有効にすることでPauli-Based Computationへプログラムを変換します
  - 現在の実装では `Dim2` のみ対応
  - 入力回路は末尾で全量子ビット測定が必要
    

## Dim2 へのコンパイル

In [ ]:
dim2_topology_path = project_root / "quration-docs" / "tutorial" / "data" / "tutorial_5_dim2_topology.yaml"
dim2_pipeline_path = project_root / "quration-docs" / "tutorial" / "data" / "tutorial_5_dim2_pipeline.yaml"

### トポロジー情報

In [ ]:
Code(filename=dim2_topology_path, language="YAML")

### パイプライン

In [ ]:
Code(filename=dim2_pipeline_path, language="YAML")

### 実行

In [ ]:
!qret compile --verbose --pipeline {dim2_pipeline_path}
!qret asm -i {output_dir / "tutorial_5_dim2.json"} -o {output_dir / "tutorial_5_dim2.asm"} --print-metadata 1

## Dim3 へのコンパイル

In [ ]:
dim3_topology_path = project_root / "quration-docs" / "tutorial" / "data" / "tutorial_5_dim3_topology.yaml"
dim3_pipeline_path = project_root / "quration-docs" / "tutorial" / "data" / "tutorial_5_dim3_pipeline.yaml"

### トポロジー情報

In [ ]:
Code(filename=dim3_topology_path, language="YAML")

### パイプライン

In [ ]:
Code(filename=dim3_pipeline_path, language="YAML")

### 実行

In [ ]:
!qret compile --verbose --pipeline {dim3_pipeline_path}
!qret asm -i {output_dir / "tutorial_5_dim3.json"} -o {output_dir / "tutorial_5_dim3.asm"} --print-metadata 1

## 分散量子計算（DistributedDim2）へのコンパイル

In [ ]:
dist_topology_path = project_root / "quration-docs" / "tutorial" / "data" / "tutorial_5_dist_topology.yaml"
dist_pipeline_path = project_root / "quration-docs" / "tutorial" / "data" / "tutorial_5_dist_pipeline.yaml"

### トポロジー情報

In [ ]:
Code(filename=dist_topology_path, language="YAML")

### パイプライン

In [ ]:
Code(filename=dist_pipeline_path, language="YAML")

### 実行

In [ ]:
!qret compile --verbose --pipeline {dist_pipeline_path}
!qret asm -i {output_dir / "tutorial_5_dist.json"} -o {output_dir / "tutorial_5_dist.asm"} --print-metadata 1

## PBC モードへのコンパイル
PBC は `sc_ls_fixed_v0_enable_pbc_mode: true` で有効化します。
このチュートリアルでは、制約を満たすように末尾で全量子ビットを測定した入力を使います。


In [ ]:
pbc_pipeline_path = project_root / "quration-docs" / "tutorial" / "data" / "tutorial_5_pbc_pipeline.yaml"
pbc_qasm_path = project_root / "quration-docs" / "tutorial" / "data" / "tutorial_5_pbc.qasm"

### 入力回路（PBC 用）

In [ ]:
Code(filename=pbc_qasm_path, language="ASM")

### パイプライン

In [ ]:
Code(filename=pbc_pipeline_path, language="YAML")

### 実行

In [ ]:
!qret compile --verbose --pipeline {pbc_pipeline_path}
!qret asm -i {output_dir / "tutorial_5_pbc.json"} -o {output_dir / "tutorial_5_pbc.asm"} --print-metadata 1

Code(filename=output_dir / "tutorial_5_pbc.asm", language="ASM")

## 次章への接続
次章では、この章で生成した `tutorial_5_*.json` を `profile` で解析し、
実行リソース（runtime・code distance・physical qubits など）を比較します。
